# Demo del registry de engine_quant (Fase 3, PLAN.md §5.4/§7.6)

Este notebook es una demo/tutorial del cliente Python (`nanobind`) sobre el registry C++
de modelos, productos y medidas del motor XVA. No repite la validación numérica del
motor (eso vive en los tests de Rust y C++, PLAN.md §5.6) — solo muestra cómo se usa la
API desde Python.

**Antes de ejecutar este notebook**, hay que compilar el proyecto con CMake
(CMake + Corrosion + `cxx` + nanobind, ver `PLAN.md` en la raíz del repo para el proceso
completo). El módulo compilado (`engine.cp3XX-....pyd` en Windows, o `engine*.so` en
Linux/Mac) queda en el directorio de build de CMake, tipicamente `build/clients/python`
relativo a la raíz del repo. Este notebook vive en `clients/python/notebooks/`, tres
niveles por debajo de la raíz, así que añadimos esa ruta relativa al `sys.path` antes
de importar `engine`.

In [1]:
!pip install pydantic plotly

In [2]:
import sys
from pathlib import Path

# Ruta relativa desde clients/python/notebooks/ hasta el directorio de build de CMake.
# Ajusta esta ruta si tu build vive en otro sitio (ej. build-debug/, o un build fuera del repo).
sys.path.insert(0, "../../../build/clients/python")
sys.path.insert(0, "../src")

import engine
import quantdesk as qd

print("módulo engine importado desde:", engine.__file__)

módulo engine importado desde: S:\Projects\engine_quant\clients\python\notebooks\../../../build/clients/python\engine.cp312-win_amd64.pyd


## 1. Arrancar el motor y listar lo que hay registrado

`engine.Engine()` envuelve el registry C++ (`Registries` + `register_builtins`,
PLAN.md §5.4): al construirlo se registran de una vez todos los modelos, productos y
medidas disponibles. `list_models()`/`list_products()`/`list_measures()` permiten
descubrir dinámicamente qué soporta el motor sin mirar el código C++ — exactamente lo
que PLAN.md §5.4 promete: "un vistazo" a todo lo que el motor soporta.

In [3]:
eng = engine.Engine()

print("modelos disponibles: ", eng.list_models())
print("productos disponibles:", eng.list_products())
print("medidas disponibles: ", eng.list_measures())

modelos disponibles:  ['HullWhite1F', 'HullWhite2F', 'GBM', 'GBM_P', 'GbmBasket']
productos disponibles: ['IRSwap', 'Payoff']
medidas disponibles:  ['ExposureProfile', 'DV01', 'UnilateralCVA', 'PayoffExposureProfileQ', 'PV', 'HullWhiteModelNpv', 'PayoffPriceQ', 'PayoffExerciseQ', 'PayoffHitProbabilityQ', 'PayoffUnilateralCvaQ', 'PayoffForecastP', 'PayoffHitProbabilityP', 'PayoffPnlDistributionP', 'PayoffSensitivityQ', 'Greek', 'PFE95', 'ExpectedExposure']


## 2. Crear un modelo: Hull-White 1 factor

`qd.HullWhite1F(a=..., b=..., sigma=..., r0=...)` (`quantdesk`, `pydantic`) construye un
objeto tipado -- `a` (velocidad de reversión), `b` (nivel de reversión de largo plazo,
constante en este caso base), `sigma` (volatilidad) y `r0` (tipo corto inicial), todos
requeridos. `.to_params()` lo traduce al mismo `Params`/dict que `create_model(nombre, params)`
ya consumía (PLAN.md §5.4) -- sin tocar el core, solo añade validación real antes de llegar
ahí (una clave mal escrita en un dict no se detecta hasta tiempo de ejecución; en `pydantic`
sí, al construir el objeto).

Esta sección y la 3 (más abajo) usan deliberadamente la fachada dinámica de bajo nivel
(`eng = engine.Engine()`, `eng.create_model(...)`) en vez de `quantdesk.Engine` -- son el
objeto de esta demo (PLAN.md §5.4: "un vistazo" al registry en sí, cómo se construye un
`Model`/`Product` desde su nombre registrado), no un flujo de pricing de negocio como el
resto de notebooks de esta batería (PLAN_API_REFACTOR.md Fase 5, `quantdesk.Engine` se usa
a partir de la sección 4, donde el objetivo pasa a ser calcular medidas, no enseñar el
registry).

In [4]:
model_spec = qd.HullWhite1F(a=0.1, b=0.03, sigma=0.01, r0=0.02)
model = eng.create_model(model_spec.model_type, model_spec.to_params())
model

<Model 'HullWhite1F'>

## 3. Crear un producto: IRS a la par a 5 años

`qd.IRSwap` necesita `notional`, `payment_times` y `accruals` (mismo largo) y un `fixed_rate`
**requerido** -- omitirlo es un `ValidationError` de `pydantic`, no un swap "a la par". Para
eso está el constructor con nombre `IRSwap.par(...)` (propuesta 2 de `PLAN_REAPI.md` §3.2):
el tipo fijo se calcula a mercado (desde la curva de `Market`, ver más abajo).

In [5]:
trade_spec = qd.IRSwap.par(
    notional=1_000_000.0,
    payment_times=[1.0, 2.0, 3.0, 4.0, 5.0],
    accruals=[1.0, 1.0, 1.0, 1.0, 1.0],
)
product = eng.create_product(trade_spec.product_type, trade_spec.to_params())
product

<Product 'IRSwap'>

## 4. Market y `quantdesk.Engine` (pricing/ejecución fijados una vez)

Antes de calcular cualquier medida hace falta un `Market` (curva de descuento observada --
`PV`/`DV01` descuentan por esta curva, `ExpectedExposure`/`PFE95`/`UnilateralCVA` solo usan
`hazard_rate`/`recovery_rate`, ver más abajo, `PLAN_REAPI.md` §6 Fase 4). El `PricingContext`
(fecha de valoración + parámetros de la simulación Monte Carlo: `n_paths`, `n_steps`, `seed`)
y el `ExecutionContext` (cómo ejecutar: `backend` `"cpu"`/`"gpu"`/`"auto"`, `precision`) ya
no se construyen ni se traducen a mano celda a celda -- `quantdesk.Engine`
(PLAN_API_REFACTOR.md §3.2) los fija UNA sola vez en el constructor; de aquí en adelante solo
hace falta un `Market` tipado para cada llamada a `.price(...)`/`.price_batch(...)`/etc.
(mismo `market_spec` reutilizado en las secciones 5, 6 y 8 de este notebook -- antes había
que reconstruir `engine.MarketSnapshot`/`PricingContext`/`ExecutionContext` a mano en cada
celda que los necesitaba, PLAN_API_REFACTOR.md §0).

In [6]:
market_spec = qd.Market(pillars=[1.0, 2.0], zero_rates=[0.02, 0.02], hazard_rate=0.02, recovery_rate=0.4)

# Un único quantdesk.Engine para el resto del notebook (secciones 5, 6 y 8): PricingContext
# (n_paths=5_000, n_steps=208, seed=7) y ExecutionContext (backend="auto") quedan fijados
# aquí una sola vez -- ningún override puntual hace falta en este notebook, ninguna celda
# de las que siguen varía n_paths/seed/backend entre sí.
qeng = qd.Engine(backend="auto", n_paths=5_000, n_steps=208, seed=7)

market_spec, qeng

(Market(pillars=[1.0, 2.0], zero_rates=[0.02, 0.02], hazard_rate=0.02, recovery_rate=0.4),
 <quantdesk.engine.Engine at 0x1ba9d7f9fd0>)

## 5. Perfil de exposición (EE/PFE) vía Monte Carlo

Las medidas `"ExpectedExposure"` y `"PFE95"` de `Engine.price` simulan el tipo corto bajo
Hull-White y revaloran el swap restante en cada trayectoria, devolviendo la exposición
esperada y el PFE al 95% en cada fecha de reseteo del propio swap (auto-derivadas del
producto, ya no un `monitoring_times` que haya que pasar a mano). `Engine.price` las calcula
juntas en una única simulación al pedirlas en el mismo lote.

In [7]:
exposure = qeng.price(trade_spec, model_spec, market_spec, ["ExpectedExposure", "PFE95"])

print("times:   ", exposure["ExpectedExposure"].times)
print("EE:      ", exposure["ExpectedExposure"].primary)
print("PFE(95%):", exposure["PFE95"].primary)

times:    [0.0, 1.0, 2.0, 3.0, 4.0]
EE:       [0.0, 12862.617942080262, 13673.529752568928, 11957.816098484913, 7124.106240024746]
PFE(95%): [0.0, 51009.920875152675, 53607.17081592724, 46152.44561612198, 27535.557267142714]


### Graficar EE/PFE

Requiere `plotly` instalado en el entorno de Jupyter (no es una dependencia del
motor en sí, solo de este notebook).

In [8]:
import plotly.graph_objects as go

COLOR_EE = "#2a78d6"   # azul: exposicion esperada (EE)
COLOR_PFE = "#eb6834"  # naranja: PFE 95%

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=exposure["ExpectedExposure"].times, y=exposure["ExpectedExposure"].primary,
    mode="lines+markers", name="EE",
    line=dict(color=COLOR_EE, width=2), marker=dict(size=7),
    hovertemplate="t=%{x:.2f} años<br>EE=%{y:,.2f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=exposure["PFE95"].times, y=exposure["PFE95"].primary,
    mode="lines+markers", name="PFE (95%)",
    line=dict(color=COLOR_PFE, width=2), marker=dict(size=7),
    hovertemplate="t=%{x:.2f} años<br>PFE95=%{y:,.2f}<extra></extra>",
))
fig.update_layout(
    title="Perfil de exposición IRS bajo Hull-White 1F",
    xaxis_title="tiempo (años)", yaxis_title="exposición",
    template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=80),
)
fig.show()

## 6. CVA unilateral

La medida `"UnilateralCVA"` de `Engine.price` reutiliza `"ExpectedExposure"` por composición
(no vuelve a correr la simulación Monte Carlo aparte, PLAN.md §7.6) y aplica la
`hazard_rate`/`recovery_rate` de `market` para llegar al CVA agregado. El resultado escalar
viene en `result.scalar` (con `result.has_scalar == True`; las medidas de perfil como
`ExpectedExposure` no rellenan este campo).

In [9]:
cva = qeng.price(trade_spec, model_spec, market_spec, ["UnilateralCVA"])

print("CVA unilateral:", cva["UnilateralCVA"].scalar)

CVA unilateral: 503.64194077997536


## 7. Calibración: ajustar un modelo a un `MarketSnapshot` (PLAN.md §7.14/§7.18)

En vez de elegir los parámetros del modelo a mano, un calibrador los ajusta a una curva
de mercado (`MarketSnapshot`, aquí fabricada con `synthetic_from_hull_white*` -- útil
para probar/demostrar calibración sin depender de datos reales). `list_calibrators()`/
`create_calibrator(nombre)`/`Calibrator.calibrate(market, estimacion_inicial)` son tan
genéricos por nombre como `create_model`/`create_product`: el motor ya tiene **dos**
calibradores, uno por modelo, y no son el mismo código con los nombres cambiados --
`HullWhite1F` calibra `a` (velocidad de reversión) y `b` (nivel de largo plazo,
cualquier signo); `HullWhite2F`/G2++ calibra `a` y `b`, las velocidades de reversión de
**ambos** factores latentes (las dos deben ser positivas). En los dos casos, `sigma`
(y en G2++ también `eta`/`rho`) se toman como datos de entrada: no están bien
identificados contra únicamente una curva de descuento (harían falta instrumentos de
volatilidad, swaptions/caps, fuera de alcance hoy).

In [10]:
print("calibradores disponibles:", eng.list_calibrators())

# --- HullWhite1F: calibra (a, b) ---
market_1f = engine.MarketSnapshot.synthetic_from_hull_white(
    a=0.15, b=0.025, sigma=0.008, r0=0.02,
    pillars=[0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0],
)
calibrator_1f = eng.create_calibrator("HullWhite1F")
# Estimación inicial deliberadamente lejos de los parámetros "verdaderos" -- tipada igual
# que cualquier otro HullWhite1F, solo que aquí sirve de punto de partida, no de resultado.
initial_guess_1f = qd.HullWhite1F(a=0.3, b=0.01, sigma=0.008, r0=0.02)
result_1f = calibrator_1f.calibrate(market_1f, initial_guess_1f.to_params())
print("HullWhite1F:", result_1f)
print("  optimal_params:", result_1f.optimal_params)

# El resultado alimenta directamente create_model -- cierra el círculo
# Mercado -> calibrar -> Modelo calibrado. qd.HullWhite1F(**...) valida el resultado antes
# de volver a pasar por create_model.
calibrated_model_1f = eng.create_model("HullWhite1F", qd.HullWhite1F(**result_1f.optimal_params).to_params())
calibrated_model_1f

calibradores disponibles: ['HullWhite1F', 'HullWhite2F']
HullWhite1F: <CalibrationResult rmse=0.000000 converged=True>
  optimal_params: {'sigma': 0.008, 'a': 0.15000000004881128, 'b': 0.02499999999890094, 'r0': 0.02}


<Model 'HullWhite1F'>

In [11]:
# --- HullWhite2F/G2++: calibra (a, b), las dos velocidades de reversión ---
market_2f = engine.MarketSnapshot.synthetic_from_hull_white_2f(
    a=0.15, b=0.25, sigma=0.008, eta=0.01, rho=-0.6, r0=0.02,
    pillars=[0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0],
)
calibrator_2f = eng.create_calibrator("HullWhite2F")
initial_guess_2f = qd.HullWhite2F(a=0.4, b=0.05, sigma=0.008, eta=0.01, rho=-0.6, r0=0.02)
result_2f = calibrator_2f.calibrate(market_2f, initial_guess_2f.to_params())
print("HullWhite2F:", result_2f)
print("  optimal_params:", result_2f.optimal_params)

calibrated_model_2f = eng.create_model("HullWhite2F", qd.HullWhite2F(**result_2f.optimal_params).to_params())
calibrated_model_2f

HullWhite2F: <CalibrationResult rmse=0.000000 converged=True>
  optimal_params: {'sigma': 0.008, 'a': 0.14999999996899865, 'b': 0.2500000001151783, 'r0': 0.02, 'eta': 0.01, 'rho': -0.6}


<Model 'HullWhite2F'>

## 8. Lotes: `price_batch`, `price_many`, `price_grid` (PLAN.md §7.17/§7.19)

Tres niveles, cada uno construido sobre el anterior -- via `quantdesk.Engine`
(PLAN_API_REFACTOR.md §3.2, Fase 2), trades/modelos/mercados tipados directamente, sin
traducir a `engine.Product`/`engine.Model`/`engine.MarketSnapshot` a mano:

- **`price_batch`** -- lote *homogéneo*: N trades del mismo tipo/calendario, vectorizado
  sin bucle escalar (el tipo corto se simula una única vez para todo el lote). Cada trade
  de `"IRSwap"` debe traer `fixed_rate` explícito -- el lote no soporta `IRSwap.par(...)`.
- **`price_many`** -- lote *heterogéneo*: admite trades de tipos/calendarios distintos, los
  agrupa internamente (por `(tipo, calendario)`) y llama a `price_batch` por grupo -- nunca
  falla por heterogeneidad.
- **`price_grid`** -- la explosión de combinaciones **Trades × Models × Markets**: por cada
  par (modelo, mercado) llama a `price_many` sobre todos los trades. `PricingContext`/
  `ExecutionContext` son compartidos (el `qeng` construido en la sección 4), no forman
  parte de la rejilla.

Las tres devuelven una fila por (trade[, modelo, mercado]) con su índice explícito -- nunca
una lista anidada -- mismo diseño en las cinco capas (C++/C ABI/Python/Excel).

In [12]:
# --- price_batch: 3 swaps del mismo calendario, cada uno con su propio notional/fixed_rate ---
schedule = {"payment_times": [1.0, 2.0, 3.0, 4.0, 5.0], "accruals": [1.0] * 5}
trade_a_spec = qd.IRSwap(notional=1_000_000.0, fixed_rate=0.02, **schedule)
trade_b_spec = qd.IRSwap(notional=2_500_000.0, fixed_rate=0.015, **schedule)
trade_c_spec = qd.IRSwap(notional=500_000.0, fixed_rate=0.025, **schedule)

batch = qeng.price_batch([trade_a_spec, trade_b_spec, trade_c_spec], model_spec, market_spec, ["PV", "UnilateralCVA"])
for row in batch:
    print(f"trade_index={row.trade_index}  PV={row.measures['PV'].scalar:.2f}  "
          f"CVA={row.measures['UnilateralCVA'].scalar:.2f}")

trade_index=0  PV=948.45  CVA=626.73
trade_index=1  PV=61254.96  CVA=2491.75
trade_index=2  PV=-11302.54  CVA=178.52


In [13]:
# --- price_many: trades de calendarios distintos, agrupados internamente ---
trade_3y_spec = qd.IRSwap(notional=2_000_000.0, fixed_rate=0.018,
                           payment_times=[1.0, 2.0, 3.0], accruals=[1.0] * 3)

# Intercalados a propósito: 5y, 3y, 5y -- dos grupos de calendario, no en bloques contiguos.
many = qeng.price_many([trade_a_spec, trade_3y_spec, trade_b_spec], model_spec, market_spec, ["PV"])
for row in many:
    print(f"trade_index={row.trade_index}  PV={row.measures['PV'].scalar:.2f}")

trade_index=0  PV=948.45
trade_index=1  PV=12691.84
trade_index=2  PV=61254.96


In [14]:
# --- price_grid: Trades x Models x Markets ---
model_2f_spec = qd.HullWhite2F(a=0.1, b=0.2, sigma=0.01, eta=0.012, rho=-0.7, r0=0.03)
market_stressed_spec = qd.Market(pillars=[1.0, 2.0], zero_rates=[0.05, 0.05], hazard_rate=0.05, recovery_rate=0.3)

grid = qeng.price_grid([trade_a_spec, trade_b_spec], [model_spec, model_2f_spec],
                        [market_spec, market_stressed_spec], ["PV", "UnilateralCVA"])
print(f"{len(grid)} celdas (2 trades x 2 modelos x 2 mercados)")
for cell in grid:
    print(f"  trade={cell.trade_index} model={cell.model_index} market={cell.market_index}  "
          f"PV={cell.measures['PV'].scalar:.2f}")

8 celdas (2 trades x 2 modelos x 2 mercados)
  trade=0 model=0 market=0  PV=948.45
  trade=1 model=0 market=0  PV=61254.96
  trade=0 model=0 market=1  PV=134913.09
  trade=1 model=0 market=1  PV=391211.55
  trade=0 model=1 market=0  PV=948.45
  trade=1 model=1 market=0  PV=61254.96
  trade=0 model=1 market=1  PV=134913.09
  trade=1 model=1 market=1  PV=391211.55


## 9. Extensibilidad del registry

El punto central de PLAN.md §5.4 es que **añadir un modelo/producto/medida nuevo no
requiere tocar los clientes**: basta con implementar la interfaz correspondiente en C++
(`IModel`/`IProduct`/`IMeasure`) y añadir una línea de registro en
`engine::register_builtins` (`cpp/engine/src/bootstrap.cpp`) -- y, si la medida nueva
debe ser invocable desde `ENGINE.PRICE`, una entrada más en la tabla de
`cpp/engine/src/price.cpp` (PLAN.md §7.15). En cuanto ese tipo nuevo existe ahí, aparece
automáticamente aquí, sin recompilar ni tocar este notebook:

In [15]:
# Vuelve a listar lo disponible: si alguien añade, por ejemplo, un modelo Black-Scholes o
# una medida FVA en cpp/engine/src/bootstrap.cpp, aparecería aquí sin cambiar este notebook.
print("modelos disponibles: ", eng.list_models())
print("productos disponibles:", eng.list_products())
print("medidas disponibles: ", eng.list_measures())

modelos disponibles:  ['HullWhite1F', 'HullWhite2F', 'GBM', 'GBM_P', 'GbmBasket']
productos disponibles: ['IRSwap', 'Payoff']
medidas disponibles:  ['ExposureProfile', 'DV01', 'UnilateralCVA', 'PayoffExposureProfileQ', 'PV', 'HullWhiteModelNpv', 'PayoffPriceQ', 'PayoffExerciseQ', 'PayoffHitProbabilityQ', 'PayoffUnilateralCvaQ', 'PayoffForecastP', 'PayoffHitProbabilityP', 'PayoffPnlDistributionP', 'PayoffSensitivityQ', 'Greek', 'PFE95', 'ExpectedExposure']
